[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_1_tools/03_numpy_for_cv.ipynb)

# Notebook 03 — NumPy for Computer Vision
### 6D Pose Vision Workshop · Part 1: Tools

**Estimated time:** 45 minutes  
**Dependencies:** `numpy`, `matplotlib`

---

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install numpy matplotlib -q
    print("Running in Google Colab — packages installed")
else:
    print("Running locally — make sure your venv is activated")

import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy version: {np.__version__}")
print(f"Python version: {sys.version.split()[0]}")

## Learning Objectives

By the end of this notebook you will:

- Understand why images are NumPy arrays and how they're structured
- Create, inspect, and manipulate arrays with confidence
- Know the critical **dtypes** for computer vision (`uint8`, `float32`, `float64`)
- Use **indexing, slicing, and masking** to work with regions of images
- Understand **broadcasting** — NumPy's most powerful and confusing feature
- Display images and plots correctly with **matplotlib**
- Know how to normalize between `[0, 255]` and `[0.0, 1.0]` without clipping errors

---

## 1. Why NumPy Is the Backbone of Computer Vision

### Images ARE arrays

When you load an image into Python, it becomes a NumPy array:

```
A 4×4 grayscale image (simplified):

pixel (0,0) = 128   pixel (0,1) = 200   pixel (0,2) = 45    pixel (0,3) = 0
pixel (1,0) = 255   pixel (1,1) = 180   pixel (1,2) = 100   pixel (1,3) = 50
pixel (2,0) = 30    pixel (2,1) = 90    pixel (2,2) = 220   pixel (2,3) = 10
pixel (3,0) = 0     pixel (3,1) = 15    pixel (3,2) = 255   pixel (3,3) = 128

In NumPy:  img.shape = (4, 4)  — 4 rows, 4 columns
           img[0, 0] = 128     — row 0, col 0
           img[2, 3] = 10      — row 2, col 3
```

A **color image** adds a third dimension (channels):

```
img.shape = (height, width, 3)  — 3 channels: Blue, Green, Red (OpenCV order)

img[row, col] = [B, G, R]   — a pixel's blue, green, red values
img[0, 0]     = [200, 100, 50]  — one pixel
img[:, :, 0]  = all blue channel values (entire B plane)
img[:, :, 2]  = all red channel values
```

Everything we do in computer vision — filtering, thresholding, detecting features, running neural networks — is ultimately **math on these arrays**. NumPy makes that math fast and readable.

In [ ]:
# ── Creating arrays ──────────────────────────────────────────────────────────

# Black image (all zeros) — 480 rows, 640 columns, 3 channels
black = np.zeros((480, 640, 3), dtype=np.uint8)
print(f"Black image:  shape={black.shape}, dtype={black.dtype}, min={black.min()}, max={black.max()}")

# White image (all 255)
white = np.ones((480, 640, 3), dtype=np.uint8) * 255
print(f"White image:  shape={white.shape}, dtype={white.dtype}, min={white.min()}, max={white.max()}")

# Grayscale gradient: values from 0 to 255 across width
gradient_row = np.linspace(0, 255, 640, dtype=np.uint8)  # 1D array: 640 values
gradient = np.tile(gradient_row, (480, 1))                # repeat 480 times as rows
print(f"Gradient:     shape={gradient.shape}, dtype={gradient.dtype}")

# From a Python list
small = np.array([[10, 20, 30],
                  [40, 50, 60],
                  [70, 80, 90]], dtype=np.uint8)
print(f"Small array:  shape={small.shape}")
print(small)

## 2. Array Shapes in CV Context

Understanding `.shape` is critical — every array operation depends on it:

| Shape | Meaning in computer vision |
|---|---|
| `(H, W)` | Grayscale image (height H, width W) |
| `(H, W, 3)` | Color image (BGR in OpenCV, RGB in matplotlib) |
| `(H, W, 4)` | Color image with alpha (transparency) channel |
| `(N, 2)` | N 2D points (e.g., detected keypoints: [[x1,y1], [x2,y2], ...]) |
| `(N, 3)` | N 3D points in object coordinates |
| `(3, 3)` | 3×3 matrix (camera intrinsics matrix K, rotation matrix R) |
| `(4, 4)` | 4×4 homogeneous transform matrix |
| `(3,)` | Translation vector, rotation vector, etc. |

In [ ]:
# Shapes you'll see constantly in this course

# Camera intrinsics matrix K (3x3)
K = np.array([[800, 0,   320],
              [0,   800, 240],
              [0,   0,   1  ]], dtype=np.float64)

# 2D image points (N x 2) — e.g., corners detected in image
image_points = np.array([[120, 80],
                         [200, 85],
                         [200, 160],
                         [120, 160]], dtype=np.float32)

# 3D object points (N x 3) — e.g., corners of a real-world object in meters
object_points = np.array([[0,    0,    0],
                          [0.05, 0,    0],
                          [0.05, 0.05, 0],
                          [0,    0.05, 0]], dtype=np.float32)

print("Camera matrix K:")
print(K)
print(f"Shape: {K.shape}")

print(f"\n2D image points: shape={image_points.shape}")
print(image_points)

print(f"\n3D object points: shape={object_points.shape}")
print(object_points)

## 3. Data Types (dtypes)

The **dtype** (data type) of an array determines:
- What values it can hold (integers vs floats, ranges)
- How much memory it uses
- Whether math operations will silently overflow or lose precision

| dtype | Range | Bytes per element | Used for |
|---|---|---|---|
| `uint8` | 0 to 255 | 1 | Images — pixel values, OpenCV default |
| `int16` | -32768 to 32767 | 2 | Intermediate image math |
| `float32` | ~-3.4e38 to 3.4e38 | 4 | Neural network weights, image preprocessing |
| `float64` | ~-1.8e308 to 1.8e308 | 8 | Camera calibration math, precise geometry |
| `bool` | True/False | 1 | Masks, thresholded images |

### The silent uint8 overflow trap

In [ ]:
# The uint8 overflow trap — very common bug in CV code

a = np.array([200, 250, 255], dtype=np.uint8)
b = np.array([100, 10,  1  ], dtype=np.uint8)

# WRONG: uint8 arithmetic wraps around silently
result_wrong = a + b
print(f"WRONG (overflow): {a} + {b} = {result_wrong}")
# 200+100=300, but 300 mod 256 = 44 — wraps around!

# CORRECT: cast to int16 first, then clip back to uint8 range
result_clipped = np.clip(a.astype(np.int16) + b.astype(np.int16), 0, 255).astype(np.uint8)
print(f"CORRECT (clipped): {a} + {b} = {result_clipped}")

# CORRECT: OpenCV provides cv2.add() which saturates at 255
# (we'll cover this in the OpenCV notebook)
print("\nKey rule: always cast to a larger dtype before arithmetic, then clip back")

In [ ]:
# Float normalization: [0, 255] uint8 ↔ [0.0, 1.0] float32

# Create a uint8 image patch
img_uint8 = np.array([[0, 128, 255],
                      [64, 192, 32]], dtype=np.uint8)

# Convert to float32 in [0, 1] range — needed for neural networks
img_float = img_uint8.astype(np.float32) / 255.0
print("uint8 image:")
print(img_uint8)
print(f"  dtype: {img_uint8.dtype}, min: {img_uint8.min()}, max: {img_uint8.max()}")

print("\nfloat32 image (normalized):")
print(img_float.round(3))
print(f"  dtype: {img_float.dtype}, min: {img_float.min():.3f}, max: {img_float.max():.3f}")

# Convert back to uint8
img_back = (img_float * 255).clip(0, 255).astype(np.uint8)
print("\nBack to uint8:")
print(img_back)
print(f"  Matches original: {np.all(img_back == img_uint8)}")

## 4. Indexing, Slicing, and Masking

### The indexing syntax: `[row, col]` or `[row, col, channel]`

```
img[0, 0]       — top-left pixel
img[-1, -1]     — bottom-right pixel
img[100, 200]   — pixel at row 100, col 200
img[100, 200, 0] — blue channel of that pixel (OpenCV BGR)
```

### Slicing: `[start:stop:step]`

```
img[0:100, :]    — first 100 rows (top strip)
img[:, 0:100]    — first 100 columns (left strip)
img[100:300, 200:400]  — rectangular region of interest (ROI)
img[::2, ::2]   — every other pixel (downsample 2×)
img[:, :, ::-1]  — reverse channel order (BGR → RGB)
```

In [ ]:
# Create a synthetic 'image' to practice indexing
img = np.zeros((300, 400, 3), dtype=np.uint8)

# Draw colored rectangles by slicing
img[0:150, 0:200] = [255, 0, 0]    # Top-left: Blue (BGR)
img[0:150, 200:400] = [0, 255, 0]  # Top-right: Green
img[150:300, 0:200] = [0, 0, 255]  # Bottom-left: Red
img[150:300, 200:400] = [0, 255, 255]  # Bottom-right: Yellow (G+R)

print(f"Image shape: {img.shape}")
print(f"Top-left pixel (BGR):     {img[75, 100]}   → displays as Red")
print(f"Top-right pixel (BGR):    {img[75, 300]}   → displays as Green")
print(f"Bottom-left pixel (BGR):  {img[225, 100]}  → displays as Blue")
print(f"Bottom-right pixel (BGR): {img[225, 300]}  → displays as Yellow")

# Display with matplotlib (BGR → RGB flip needed!)
plt.figure(figsize=(8, 6))
plt.imshow(img[:, :, ::-1])  # ::-1 reverses channel order BGR → RGB
plt.title("Sliced color regions — note BGR→RGB flip for display")
plt.axis('off')
plt.show()

In [ ]:
# Boolean masking — extremely useful for thresholding and segmentation

gray = np.array([[10, 150, 200, 30],
                 [90, 180, 50,  220],
                 [255, 5,  120, 80]], dtype=np.uint8)

# Create a mask: True where pixel value > 100
mask = gray > 100
print("Original array:")
print(gray)
print("\nMask (> 100):")
print(mask)

# Use mask to extract values
bright_pixels = gray[mask]
print(f"\nBright pixel values (>{100}): {bright_pixels}")

# Use mask to set values (zero out dark pixels)
result = gray.copy()
result[~mask] = 0  # ~ inverts the mask
print("\nDark pixels zeroed out:")
print(result)

## 5. Broadcasting — NumPy's Superpower

### The intuition

Broadcasting lets you do math between arrays of **different shapes** — NumPy automatically "stretches" the smaller array to match the larger one.

**Analogy:** Imagine a spreadsheet. You have a column of numbers and you want to add 10 to every row. You don't need to copy "10" into a whole column — you just add 10 to the column directly.

Broadcasting is that, generalized to any dimension.

### Broadcasting rules (simplified)

Two shapes are compatible if, reading from the **right**:
1. The dimensions are equal, OR
2. One of them is 1

```
Shape (480, 640, 3) + Shape (3,) → broadcasts to (480, 640, 3)
  [img]               [value for each channel]

Shape (480, 640) + Shape (480, 1) → broadcasts to (480, 640)
  [image]             [one value per row]

Shape (480, 640, 3) + Shape (480, 640) → ERROR (last dim: 3 vs 640)
```

In [ ]:
# Broadcasting examples for computer vision

img = np.zeros((100, 100, 3), dtype=np.float32)

# Example 1: Add a per-channel bias (e.g., adjust brightness per channel)
# Image: (100, 100, 3) + bias: (3,) → broadcasts to (100, 100, 3)
bias = np.array([0.1, 0.2, 0.05])  # add 0.1 to B, 0.2 to G, 0.05 to R
img_biased = np.clip(img + bias, 0.0, 1.0)
print(f"Image shape: {img.shape}, Bias shape: {bias.shape}")
print(f"Result shape: {img_biased.shape}")
print(f"Top-left pixel after bias: {img_biased[0,0]}")

# Example 2: ImageNet normalization (standard preprocessing for deep learning)
# Subtract per-channel mean, divide by per-channel std
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])  # RGB means
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])  # RGB stds

# Simulate a normalized float image in [0, 1]
img_rgb = np.random.rand(100, 100, 3).astype(np.float32)

# Broadcasting handles this elegantly:
img_normalized = (img_rgb - IMAGENET_MEAN) / IMAGENET_STD
print(f"\nImageNet normalized: mean~{img_normalized.mean():.2f}, std~{img_normalized.std():.2f}")

# Example 3: Row-wise normalization (normalize each row to [0, 1])
data = np.array([[10, 20, 30],
                 [100, 50, 200],
                 [5, 5, 5]], dtype=np.float32)

row_mins = data.min(axis=1, keepdims=True)   # shape (3, 1) — one min per row
row_maxs = data.max(axis=1, keepdims=True)   # shape (3, 1)
data_norm = (data - row_mins) / (row_maxs - row_mins + 1e-8)
print("\nOriginal:", data)
print("Row-normalized:", data_norm)

## 6. Reshaping and Stacking

In [ ]:
# Reshaping — change the layout without changing the data

# Flatten an image to 1D (common before feeding to ML models)
small_img = np.arange(12, dtype=np.uint8).reshape(3, 4)  # 3 rows, 4 cols
print("Original (3, 4):")
print(small_img)

flattened = small_img.flatten()  # or small_img.ravel() for a view
print(f"\nFlattened (12,): {flattened}")

# Reshape back
restored = flattened.reshape(3, 4)
print(f"\nRestored (3, 4):")
print(restored)

# Useful reshape patterns
img = np.zeros((480, 640, 3), dtype=np.uint8)
print(f"\nImage shape:          {img.shape}")
print(f"Add batch dimension:  {img[np.newaxis].shape}")    # (1, 480, 640, 3)
print(f"Squeeze batch:        {img[np.newaxis].squeeze().shape}")  # back to (480, 640, 3)
print(f"Flatten HxW pixels:   {img.reshape(-1, 3).shape}") # (307200, 3) — each pixel

In [ ]:
# Stacking — combining arrays

# Split a color image into channels and recombine
h, w = 100, 100
blue  = np.zeros((h, w), dtype=np.uint8)
green = np.full((h, w), 128, dtype=np.uint8)
red   = np.full((h, w), 200, dtype=np.uint8)

# Stack along a new axis (axis=2 = channel axis)
bgr_img = np.stack([blue, green, red], axis=2)
print(f"Stacked BGR image: {bgr_img.shape}")
print(f"Pixel [0,0] BGR: {bgr_img[0,0]}")

# np.concatenate — join along existing axis
left  = np.zeros((100, 200, 3), dtype=np.uint8)
right = np.ones((100, 200, 3), dtype=np.uint8) * 128
combined = np.concatenate([left, right], axis=1)  # join along columns
print(f"\nConcatenated side-by-side: {combined.shape}")  # (100, 400, 3)

# np.vstack / np.hstack — shortcuts
top    = np.zeros((50, 100), dtype=np.uint8)
bottom = np.ones((50, 100), dtype=np.uint8) * 200
stacked = np.vstack([top, bottom])  # stack vertically
print(f"Vstacked:          {stacked.shape}")  # (100, 100)

## 7. Displaying Images with Matplotlib

### The two display modes for images

```python
plt.imshow(img)              # color image — expects RGB (NOT BGR!)
plt.imshow(img, cmap='gray') # grayscale — expects 2D array (no channels)
```

### The BGR → RGB gotcha

OpenCV stores color images as **BGR** (Blue first). Matplotlib expects **RGB** (Red first). If you don't convert, colors look wrong: red objects appear blue.

Fix: `img[:, :, ::-1]` reverses the channel order.

In [ ]:
# Demonstrate BGR vs RGB confusion and how to fix it

# Create an image with a clearly red square (BGR: R channel = index 2)
img_bgr = np.zeros((200, 200, 3), dtype=np.uint8)
img_bgr[50:150, 50:150, 2] = 255  # set R channel (index 2 in BGR) to max

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Wrong: display BGR as-is (matplotlib interprets it as RGB)
axes[0].imshow(img_bgr)
axes[0].set_title('WRONG: BGR displayed as RGB\n(red square appears BLUE)')
axes[0].axis('off')

# Correct: convert BGR → RGB before display
img_rgb = img_bgr[:, :, ::-1]
axes[1].imshow(img_rgb)
axes[1].set_title('CORRECT: BGR→RGB converted\n(red square appears RED)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Useful matplotlib patterns for CV work

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Grayscale image
gray = np.zeros((100, 200), dtype=np.uint8)
for i in range(200):
    gray[:, i] = int(i / 200 * 255)
axes[0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Grayscale gradient')
axes[0].axis('off')
plt.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046)

# 2. Heatmap (useful for depth maps, score maps)
depth_sim = np.random.exponential(scale=1.5, size=(100, 200)).astype(np.float32)
im = axes[1].imshow(depth_sim, cmap='plasma')
axes[1].set_title('Simulated depth map (plasma colormap)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# 3. Float image in [0, 1] — note: vmin/vmax needed!
float_img = np.random.rand(100, 200, 3).astype(np.float32)
axes[2].imshow(float_img)
axes[2].set_title('Random float RGB image [0.0, 1.0]')
axes[2].axis('off')

plt.suptitle('Common image display patterns in matplotlib', y=1.02)
plt.tight_layout()
plt.show()

## 8. Key Array Operations for Computer Vision

In [ ]:
# Statistical operations
img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

print("Per-channel statistics (common in camera calibration and image analysis):")
for i, channel in enumerate(['Blue', 'Green', 'Red']):
    ch = img[:, :, i]
    print(f"  {channel}: mean={ch.mean():.1f}, std={ch.std():.1f}, "
          f"min={ch.min()}, max={ch.max()}")

# Matrix operations (critical for camera math)
print("\nMatrix operations:")

# Camera intrinsics matrix
K = np.array([[800., 0., 320.],
              [0., 800., 240.],
              [0.,  0.,   1.]])

# Rotation matrix (90° around Z axis)
theta = np.pi / 2
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0,              0,             1]])

print(f"K shape: {K.shape}, R shape: {R.shape}")
print(f"K @ R (matrix product) shape: {(K @ R).shape}")
print(f"det(R) = {np.linalg.det(R):.4f} (should be 1.0 for rotation matrices)")
print(f"R.T @ R ≈ I ? {np.allclose(R.T @ R, np.eye(3), atol=1e-10)} (orthogonality check)")

## Recap

| Concept | Key takeaway |
|---|---|
| Images = arrays | Grayscale: `(H,W)`, Color: `(H,W,3)`, Batch: `(N,H,W,3)` |
| Array creation | `np.zeros`, `np.ones`, `np.linspace`, `np.arange`, `np.array` |
| dtype | `uint8` for pixels (0–255), `float32` for ML, `float64` for math |
| uint8 overflow | Cast to `int16` before arithmetic, then `clip(0,255).astype(uint8)` |
| Indexing | `img[row, col]`, `img[r1:r2, c1:c2]` for ROI, `img[:,:,::-1]` for BGR→RGB |
| Broadcasting | NumPy auto-stretches shapes — read rules from the right |
| matplotlib | Use `cmap='gray'` for 2D; convert BGR→RGB for color images |
| Normalization | `float = uint8 / 255.0`, reverse: `(float * 255).clip(0,255).astype(uint8)` |

---

**Next:** `04_intro_to_opencv.ipynb` — use NumPy arrays with OpenCV to load, display, and work with real images.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Create a gradient image
# ============================================================
# Create a 200×300 uint8 image where:
#   - Pixel values increase LEFT to RIGHT (0 on left, 255 on right)
#   - This creates a horizontal gradient
#   - Display it as grayscale with matplotlib
#
# Hint: use np.linspace and np.tile

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# row = np.linspace(0, 255, 300, dtype=np.uint8)  # 300 values across width
# gradient = np.tile(row, (200, 1))               # repeat for 200 rows
# print(f"Shape: {gradient.shape}, dtype: {gradient.dtype}")
# print(f"Left edge: {gradient[100, 0]}, Right edge: {gradient[100, 299]}")
# plt.figure(figsize=(8, 3))
# plt.imshow(gradient, cmap='gray', vmin=0, vmax=255)
# plt.title('Horizontal gradient: 0 (left) → 255 (right)')
# plt.colorbar()
# plt.axis('off')
# plt.show()

In [ ]:
# ============================================================
# EXERCISE 2: Reshape and inspect
# ============================================================
# Given a flattened array representing a 28×28 grayscale image:
#   1. Reshape it to (28, 28)
#   2. Print: shape, dtype, min, max, mean
#   3. Extract the center 14×14 region
#   4. Display the full image and the center crop side by side

# Simulate a flat MNIST-style array (values 0-255)
np.random.seed(42)
flat_image = np.random.randint(0, 256, 784, dtype=np.uint8)  # 28*28 = 784

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# img_28 = flat_image.reshape(28, 28)
# print(f"Shape: {img_28.shape}")
# print(f"dtype: {img_28.dtype}")
# print(f"min={img_28.min()}, max={img_28.max()}, mean={img_28.mean():.1f}")
#
# # Center 14x14 region
# center = img_28[7:21, 7:21]
# print(f"Center crop shape: {center.shape}")
#
# fig, axes = plt.subplots(1, 2, figsize=(8, 4))
# axes[0].imshow(img_28, cmap='gray', vmin=0, vmax=255)
# axes[0].set_title('Full 28×28 image')
# axes[0].axis('off')
# axes[1].imshow(center, cmap='gray', vmin=0, vmax=255)
# axes[1].set_title('Center 14×14 crop')
# axes[1].axis('off')
# plt.tight_layout()
# plt.show()

In [ ]:
# ============================================================
# EXERCISE 3: Channel statistics on a synthetic color image
# ============================================================
# Create a 100×100 BGR color image where:
#   - Left half: pure blue (B=255, G=0, R=0)
#   - Right half: pure red (B=0, G=0, R=255)
#
# Then compute and print:
#   - Mean value of each channel (B, G, R) for the full image
#   - Which channel has the highest mean?
#   - Display the image (remember BGR→RGB for matplotlib!)

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# img = np.zeros((100, 100, 3), dtype=np.uint8)
# img[:, :50, 0] = 255   # left half: Blue channel
# img[:, 50:, 2] = 255   # right half: Red channel
#
# for i, name in enumerate(['Blue', 'Green', 'Red']):
#     print(f"{name} channel mean: {img[:,:,i].mean():.1f}")
#
# means = [img[:,:,i].mean() for i in range(3)]
# max_ch = ['Blue', 'Green', 'Red'][np.argmax(means)]
# print(f"Highest mean channel: {max_ch}")
# # Blue mean ≈ 127.5 (half the image), Red mean ≈ 127.5, Green ≈ 0.0
# # Both Blue and Red are equal since exactly half the image each.
#
# plt.figure(figsize=(5, 5))
# plt.imshow(img[:, :, ::-1])  # BGR → RGB
# plt.title('Left: Blue, Right: Red (shown in RGB)')
# plt.axis('off')
# plt.show()